In [12]:
# Model
from pprint import pprint

from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://host.docker.internal:11434",
    keep_alive="12m",
    model="gemma3:1b",
    # num_ctx=8192,  # cap KV cache to 8K — qwen3 defaults to 40K which wastes memory on 16GB RAM
    # num_predict=1024,  # reasonable ceiling; -1 (unlimited) is risky with thinking models
    num_thread=8,  # match available CPU cores
    # reasoning=False,  # disable thinking mode — maps to think=False in the Ollama API
    # repeat_penalty=1.0,  # disable repeat penalty — default (1.1) causes small models to stop early
    temperature=0.48,
)
pprint(llm)


ChatOllama(model='gemma3:1b', num_thread=8, temperature=0.48, keep_alive='12m', base_url='http://host.docker.internal:11434')


In [14]:
# Run Model
from pprint import pprint

hello_world = llm.invoke("Hello, world!")
hello_world.pretty_print()
pprint(hello_world.model_dump(exclude={"content"}))

================================== Ai Message ==================================

Hello there! 👋 How can I help you today?
{'additional_kwargs': {},
 'id': 'lc_run--019cbc2c-b4bf-7ea1-aaae-7a8e7f54e9bd-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'created_at': '2026-03-05T04:06:12.774463Z',
                       'done': True,
                       'done_reason': 'stop',
                       'eval_count': 12,
                       'eval_duration': 132133085,
                       'load_duration': 178817916,
                       'logprobs': None,
                       'model': 'gemma3:1b',
                       'model_name': 'gemma3:1b',
                       'model_provider': 'ollama',
                       'prompt_eval_count': 13,
                       'prompt_eval_duration': 18304416,
                       'total_duration': 355757083},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_tokens': 13,
                    'output_tokens': 12

In [9]:
# Tools
import trafilatura
from langchain_community.utilities import SearxSearchWrapper
from langchain_core.tools import tool

searxng_wrapper = SearxSearchWrapper(searx_host="http://searxng:8080")


@tool
def searxng_search(query: str) -> str:
    """Search using SearxNG, a self-hosted metasearch engine aggregating
    results from various sources. Fetches and returns the full text content
    of the top result pages for comprehensive research.

    Args:
        query: The search terms to look up, phrased as a concise natural language query.
    """
    searxng_results = searxng_wrapper.results(query, num_results=12)
    trafilatura_results = []
    for i, result in enumerate(searxng_results, start=1):
        url = result.get("link", "")
        title = result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    return "\n\n---\n\n".join(trafilatura_results)


# tool_results = searxng_search.invoke("What is the meaning of the tarot card 'The Hermit'?")
# pprint(tool_results)

In [18]:
# Agent
from pathlib import Path

from langchain_core.messages import HumanMessage, ToolMessage

from src.subjects import TAROT_CARDS


def ingest_article(subject):
    tool_message = ToolMessage(
        content=searxng_search.invoke(
            f'"{subject["name"]}" {subject["category"]["name"]} meaning significance interpretation symbolism themes correspondences'
        ),
        tool_call_id="searxng_search",
    )
    tool_message.pretty_print()

    human_message = HumanMessage(
        content=f"""\
    Using only the provided search results, write a comprehensive reference article about **{subject["name"]}** ({subject["category"]["name"]}). Target length: 1200 words.

    Focus mainly on the archetypal meaning, significance, and interpretation.
    Highlight its key symbols and themes, and what they represent and reveal.
    Explain how its energy applies to different aspects of life and the world.
    Describe its place in the broader category of {subject["category"]["name"]}.
    Note its traditional correspondences (astrology, element, number, chakra, sephiroth, etc.).
    Optimize the output article structure for use in llm agent contexts.

    Use only the provided search results as source material, and output only the article contents without any additional commentary or explanation.
    """
    )
    human_message.pretty_print()

    agent_message = llm.invoke([tool_message, human_message])
    agent_message.pretty_print()

    output_directory = Path(f"../../output/{subject['category']['slug']}")
    output_directory.mkdir(parents=True, exist_ok=True)
    output_filename = f"{subject['order']}-{subject['slug']}.md"
    output_path = output_directory / output_filename
    output_path.write_text(agent_message.content, encoding="utf-8")
    print(f"\nWritten to: {output_path.resolve()}\n")


for subject in TAROT_CARDS[1:2]:
    ingest_article(subject)

================================= Tool Message =================================

## Search Result 1: 10分钟了解英语中定冠词 the 的用法 ——7天拿下英语高分（强学国际）

**Source:** [https://www.zhihu.com/tardis/bd/art/397207179](https://www.zhihu.com/tardis/bd/art/397207179)

定冠词 the 的用法
1)特指某(些)人或某(些)事物。如:
This is the house where Lu Xun once lived.这是鲁迅曾经住过的房子。(以别于其他房子)
The book on the desk is an English dictionary.书桌上的那本书是一本英语词典。 (特指桌上的那本书。注意名词 book 被 on the desk 短语所限定。)
Cairo is the capital of Egypt.开罗是埃及的首都。
We plan to cut the wheat in these fields in three days' time.我们计划三天后割这些地的小麦。(特指这些地里的小麦)
2)指说话人与听话人彼此所熟悉的人或事物。如:
Open the door, please.请开门。(双方都知道指的是哪一个门)
The Manager is in his office.经理在他的办公室里。(双方都知道指的是哪个经理)
Let's meet at the railway station.我们在火车站碰头吧。(双方都知道指的是哪一个火车站)
3)复述上文提过的人或事物。如:
Last week I read a story and a play. The story is about the Second World War and the play about the life of university students.上周我读了一篇故事和一个剧本。那篇故事是关于第二次世界大战的，剧本是关于大学生生活的。
4)表示在世界上独一无二的事物，如
the sun 太阳，the moon 月亮，the earth 地球，